# DCAT-AP-NL Requirement Analysis for Dataverse Metadata Exporter



In [316]:
# boiler plate functions
# imports SPARQL prefixes and functions defs
import csv
from pprint import pprint
# from SPARQLWrapper import SPARQLWrapper, JSON, TURTLE, CSV 
from rdflib import Graph

prefixes = '''    
PREFIX adms: <http://www.w3.org/ns/adms#>
PREFIX dct: <http://purl.org/dc/terms/>
PREFIX dcat: <http://www.w3.org/ns/dcat#>
PREFIX dcatap: <http://data.europa.eu/r5r/>
PREFIX eli: <http://data.europa.eu/eli/ontology#>
PREFIX eush: <https://purl.eu/ns/shacl#>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
PREFIX prov: <http://www.w3.org/ns/prov#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX sh: <http://www.w3.org/ns/shacl#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX vcard: <http://www.w3.org/2006/vcard/ns#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

PREFIX dcatapnl-sh: <http://modellen.geostandaarden.nl/dcat-ap-nl/id/shape/>
'''    

def sparql_files_query(query, format, filespath):
    g = Graph()
    for filepath in filespath:
        g.parse(filepath, format=format)  # can also use "ttl" for Turtle
    query = prefixes + query
    results = g.query(query)
    return results

def create_csv(filepath, headers, data_dict):

    with open(filepath, 'w', newline='') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=headers)
        writer.writeheader()
        writer.writerows(data_dict)


# def sparql_query(query, format):
#     formats = {"json": JSON, "turtle": TURTLE, "csv": CSV}
#     f_ = formats[format]
#     endpoint = "http://vocab.getty.edu/sparql"
#     sparql = SPARQLWrapper(endpoint)
#     query = prefixes + query     
#     sparql.setQuery(query)
#     sparql.setReturnFormat(f_)
#     results = sparql.query().convert()    
#     return results


# def print_sparql_results(results):
#     for row in results["results"]["bindings"]:
#         return (row)


# Requirement: Mandatory Dataset properties - Research

**What the mandatory properties of dcat:Dataset in DCAT-AP?**
ie. `dct:description`

**What the mandatory properties of dcat:Dataset in DCAT-AP-NL?**
ie. `dct:identifier`

* include cardinality and property range


According to https://docs.geostandaarden.nl/dcat/dcat-ap-nl30/#17C1E0BE
> The SHACL rules of DCAT-AP-NL build on the SHACL rules from DCAT API-3.0. **All the rules from DCAT-AP-3.0 (see [DCAT AP-3.0 dcat-ap-SHACL.ttl](dcat-ap/releases/3.0.0/shacl/dcat-ap-SHACL.ttl)) are still applicable. DCAT-AP-NL only tightens some data rules.**

>To test whether a dataset description meets DCAT-AP-NL, it is also necessary to include both the DCAT-AP SHACL shapes and the DCAT-AP-NL SHACL shapes in the validation.

> To properly support the validation of the dataset descriptions, DCAT-AP breaks the SHACL shapes is split to support different validation scenarios and aspects. See the chapter **[Validation of DCAT-AP](https://semiceu.github.io/DCAT-AP/releases/3.0.0/#validation-of-dcat-ap)**.

**DCAT-AP-NL SHACL shapes** are divided into:

* [dcat-ap-nl/shapes/dcat-ap-nl-SHACL.ttl](dcat-ap-nl-SHACL.ttl): The SHACL shapes of DCAT-AP-NL, excluding the validation rules around the class range of properties.
* [dcat-ap-nl/shapes/dcat-ap-nl-SHACL-klassebereik.ttl](dcat-ap-nl-SHACL-klassebereik.ttl) The SHACL shapes of DCAT-AP-EN for validating the class range of properties, excluding the class range of properties with a value derived from a code list.
* [dcat-ap-nl/shapes/dcat-ap-nl-SHACL-klassebereik-codelijsten.ttl](dcat-ap-nl-SHACL-klassebereik-codelijsten.ttl:) dcat-ap-nl-SHACL class range-codelists.ttl : The SHACL shapes of DCAT-AP-EN for validating the class range of properties with a value from a code list.
* [dcat-ap-nl/shapes/dcat-ap-nl-SHACL-aanbevolen.ttl](dcat-ap-nl-SHACL-aanbevolen.ttl): The SHACL shapes of DCAT-AP-EN for validating recommended properties.



In [317]:
# INVESTIGATION
# Goal: understand how the cardinality 1..(mandatory) is expressed in DCAT-AP shacl
# By: SPARQL DESCRIBE of dcat-ap-SHACL.ttl dcat:Dataset: dct:description  shape in DCAT-AP SHACL
# Answer: via property:value  shacl:minCount 1 ;

sparql_dcatap_dataset_1mandatory_props = ''' 
DESCRIBE ?prop_shape
WHERE {
    BIND(dct:description AS ?prop_path) .
    <https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DatasetShape> sh:property ?prop_shape .
    ?prop_shape sh:path ?prop_path .
}
'''
results = sparql_files_query(query=sparql_dcatap_dataset_1mandatory_props, 
                            format='ttl', 
                            filespath=['dcat-ap/releases/3.0.1/shacl/dcat-ap-SHACL.ttl'])
print(results.serialize(format='ttl').decode('utf-8'))


@prefix dc1: <http://purl.org/dc/terms/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix shacl: <http://www.w3.org/ns/shacl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DatasetShape/6cae9880e515253132af1452a38a8a5827165149> rdfs:seeAlso "https://semiceu.github.io/DCAT-AP/releases/3.0.1#Dataset.description" ;
    shacl:description "A free-text account of the Dataset."@en ;
    shacl:name "description"@en ;
    shacl:nodeKind shacl:Literal ;
    shacl:path dc1:description .

<https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DatasetShape/8f61614144aa7bca188b24f5976593dc08aad0e6> rdfs:seeAlso "https://semiceu.github.io/DCAT-AP/releases/3.0.1#Dataset.description" ;
    shacl:description "A free-text account of the Dataset."@en ;
    shacl:minCount 1 ;
    shacl:name "description"@en ;
    shacl:path dc1:description .




The pattern I see in the cell above (`dcat:Dataset: dct:description`) property shapes, makes me conclude that 
* in **DCAT-AP SHACL the required properties have `shacl:minCount 1`**, which makes sense

Follow-up questions/queries?

* which other Dataset properties have `shacl:minCount 1` AKA are mandatory? 
* is the same pattern present in DCAT-AP-NL shacl?

In [318]:
# OUTPUT
# Goal: list of all dcat:Dataset mandatory properties in DCAT-AP & DCAT-APN-NL
# shacl:minCount 1

sparql_apnl_dataset_mandatory_props = ''' 
SELECT ?prop_path  ?prop_shape ?maxCount
WHERE {
    {   # DCAT-AP query
        <https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DatasetShape> sh:property ?prop_shape .
        ?prop_shape sh:minCount 1 ;
            sh:path ?prop_path .
    }
    UNION
    {   # DCAT-AP-NL query
        dcatapnl-sh:DatasetShape sh:property ?prop_shape .
        ?prop_shape sh:minCount 1 ;
            sh:path ?prop_path . 
    }
}
'''


results_ap_nl = sparql_files_query(query=sparql_apnl_dataset_mandatory_props, 
                            format='ttl', 
                            filespath=[ 
                                'dcat-ap-nl/shapes/dcat-ap-nl-SHACL.ttl',
                                'dcat-ap/releases/3.0.1/shacl/dcat-ap-SHACL.ttl'])

print(f"{'*'*10} Dataset Mandatory Properties in DCAT-AP & DCAT-AP-NL  {'*'*10}")
for row in results_ap_nl:
    pprint(row.asdict())

ap_NL_mandatory_dataset_props_list = [row.asdict() for row in results_ap_nl]

# create_csv(filepath='dcat-ap-nl_mand_props.csv',
#            headers=ap_NL_mandatory_dataset_props_list[0].keys(),
#            data_dict=ap_NL_mandatory_dataset_props_list)


********** Dataset Mandatory Properties in DCAT-AP & DCAT-AP-NL  **********
{'prop_path': rdflib.term.URIRef('http://purl.org/dc/terms/description'),
 'prop_shape': rdflib.term.URIRef('https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DatasetShape/8f61614144aa7bca188b24f5976593dc08aad0e6')}
{'prop_path': rdflib.term.URIRef('http://purl.org/dc/terms/title'),
 'prop_shape': rdflib.term.URIRef('https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DatasetShape/f8b02efd063b8089b72ce9677a1e4a3488eeb9a9')}
{'prop_path': rdflib.term.URIRef('http://purl.org/dc/terms/accessRights'),
 'prop_shape': rdflib.term.URIRef('http://modellen.geostandaarden.nl/dcat-ap-nl/id/shape/DatasetShape_accessRights_minCount')}
{'prop_path': rdflib.term.URIRef('http://www.w3.org/ns/dcat#contactPoint'),
 'prop_shape': rdflib.term.URIRef('http://modellen.geostandaarden.nl/dcat-ap-nl/id/shape/DatasetShape_contactPoint_minCount')}
{'prop_path': rdflib.term.URIRef('http://purl.org/dc/terms/creator'),
 'prop_shape': 

# Requirement: Mandatory Dataset properties

The response to the query above, tells us that the mandatory Dataset properties are:

See [csvs/ap-nl-dataset-mand-props.csv](csvs/ap-nl-dataset-mand-props.csv) where this info is compiled

**DCAT-AP mandatory properties of dcat:Dataset:**

*  http://purl.org/dc/terms/description
*  http://purl.org/dc/terms/title 

**DCAT-AP-NL mandatory properties of dcat:Dataset:**

* http://purl.org/dc/terms/accessRights 
* http://www.w3.org/ns/dcat#contactPoint 
* http://purl.org/dc/terms/creator 
* http://purl.org/dc/terms/identifier 
* http://purl.org/dc/terms/publisher 
* http://www.w3.org/ns/dcat#theme


# Requirement Enunciation: **Range** of DCAT-AP + DCAT-AP-NL Mandatory Properties

The range is the type of values a property can have.

The focus here is to **find the ranges of *object properties* (that have other RDF nodes as their value)** in n [dcat-ap/releases/3.0.1/shacl/ranges.ttl](dcat-ap/releases/3.0.1/shacl/ranges.ttl). *Data properties*, that have strings or numbers as values, are not being address by ranges.ttl.


Example of Dataset dcat:creator property and its range description defined by `shacl:class foaf:Agent`

```
<https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DatasetShape/c1d40f7102c8201949576e76be48b991b47958d9> rdfs:seeAlso "https://semiceu.github.io/DCAT-AP/releases/3.0.1#Dataset.creator";
  shacl:class foaf:Agent;
  shacl:description "An entity responsible for producing the dataset."@en;
  shacl:name "creator"@en;
  shacl:path dc:creator .
```




In [319]:
# Goal: info **Ranges** of Dataset mandatory object properties
# note: object properties have as values RDF nodes (class instances), in contrast with data properties which have literals as values
# Output: Dataset mandatory object properties shapes - 
# property in schal:path; range in shacl:class
#  

prop_path_for_sparql = [item['prop_path'] for item in ap_NL_mandatory_dataset_props_list]
prop_path_for_sparql_str4query = (' '.join([f'<{str(uri)}>' for uri in prop_path_for_sparql]))
# use prop_path_for_sparql_str4query as VALUES in following range query
sparql_dcat_mandatory_dataset_props_ranges = '''
DESCRIBE  ?prop_shape_uri
WHERE {
    BIND( dcat:Dataset AS ?targetClass )
    VALUES ?prop_path { %s } 
    ?shape a sh:NodeShape ;
           sh:targetClass ?targetClass ;
           sh:property ?prop_shape_uri .
    ?prop_shape_uri sh:path ?prop_path  .
    
}''' % prop_path_for_sparql_str4query
print(f"{'*'*3} Querying shapes for Dataset mandatory propreties: {prop_path_for_sparql_str4query} {'*'*3}\n")
describe_dcatap_NL_mandatory_dataset_props = sparql_files_query(query=sparql_dcat_mandatory_dataset_props_ranges, 
                                                          format='ttl', 
                                                          filespath=['dcat-ap/releases/3.0.1/shacl/ranges.ttl'])
print(describe_dcatap_NL_mandatory_dataset_props.serialize(format='ttl').decode('utf-8'))

# TODO: include this info in the CSV dcat-ap-nl_mand_props.csv

*** Querying shapes for Dataset mandatory propreties: <http://purl.org/dc/terms/description> <http://purl.org/dc/terms/title> <http://purl.org/dc/terms/accessRights> <http://www.w3.org/ns/dcat#contactPoint> <http://purl.org/dc/terms/creator> <http://purl.org/dc/terms/identifier> <http://purl.org/dc/terms/publisher> <http://www.w3.org/ns/dcat#theme> ***

@prefix dc1: <http://purl.org/dc/terms/> .
@prefix dcat: <http://www.w3.org/ns/dcat#> .
@prefix foaf: <http://xmlns.com/foaf/0.1/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix shacl: <http://www.w3.org/ns/shacl#> .
@prefix skos: <http://www.w3.org/2004/02/skos/core#> .
@prefix vcard: <http://www.w3.org/2006/vcard/ns#> .

<https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DatasetShape/7b6713c1f4a52e964f5db57eabef294b6d04e90e> rdfs:seeAlso "https://semiceu.github.io/DCAT-AP/releases/3.0.1#Dataset.contactpoint" ;
    shacl:class vcard:Kind ;
    shacl:description "Contact information that can be used for sending co

In [320]:
# continuation of previous cell 
# Output:  for DCAT-AP-NL Dataset mandatory properties and their range (sh:class ?prop_range) 
# Output: ap-nl-dataset-mand-props.csv

print(f'{"-"*40}\nQuerying in dcat-ap/releases/3.0.1/shacl/ranges.ttl Dataset property shapes:\n{prop_path_for_sparql_str4query}\n{"-"*40}')

sparql_dcat_mandatory_distribution_props_ranges_vars = '''
SELECT  ?prop_path  ?prop_range
WHERE {
    BIND( dcat:Dataset AS ?targetClass )
    VALUES ?prop_path { %s } 
    ?shape a sh:NodeShape ;
           sh:targetClass ?targetClass ;
           sh:property ?prop_shape_uri .
    ?prop_shape_uri sh:path ?prop_path ;
                    sh:class ?prop_range .
    
}''' % prop_path_for_sparql_str4query


ap_NL_mandatory_dataset_props_range = sparql_files_query(query=sparql_dcat_mandatory_distribution_props_ranges_vars, 
                                                          format='ttl', 
                                                          filespath=['dcat-ap/releases/3.0.1/shacl/ranges.ttl'])
ap_NL_mandatory_dataset_props_range_list = [row.asdict() for row in ap_NL_mandatory_dataset_props_range]

# Join  ap_NL_mandatory_dataset_props_range_list & ap_NL_mandatory_dataset_props_list
# Step 1: Build a mapping from prop_path to prop_range
prop_range_map = {d['prop_path']: d['prop_range'] for d in ap_NL_mandatory_dataset_props_range_list}
# Step 2: Merge the lists
ap_NL_mandatory_dataset_props_merged = []
for d in ap_NL_mandatory_dataset_props_list:
    # Copy to avoid mutating the original
    merged_dict = d.copy()
    prop_path = d['prop_path']
    if prop_path in prop_range_map:
        merged_dict['prop_range'] = prop_range_map[prop_path]
    ap_NL_mandatory_dataset_props_merged.append(merged_dict)
print(f"{'*'*3} Output of ap_NL_mandatory_dataset_props_merged in ap-nl-dataset-mand-props.csv {'*'*3}\n")
# print(ap_NL_mandatory_dataset_props_merged)
create_csv(filepath='csvs/ap-nl-dataset-mand-props.csv',
           headers=['prop_path', 'prop_range', 'prop_shape'],
           data_dict=ap_NL_mandatory_dataset_props_merged)


----------------------------------------
Querying in dcat-ap/releases/3.0.1/shacl/ranges.ttl Dataset property shapes:
<http://purl.org/dc/terms/description> <http://purl.org/dc/terms/title> <http://purl.org/dc/terms/accessRights> <http://www.w3.org/ns/dcat#contactPoint> <http://purl.org/dc/terms/creator> <http://purl.org/dc/terms/identifier> <http://purl.org/dc/terms/publisher> <http://www.w3.org/ns/dcat#theme>
----------------------------------------
*** Output of ap_NL_mandatory_dataset_props_merged in ap-nl-dataset-mand-props.csv ***



In [321]:
%%!
echo "---- DCAT-AP-NL - mandatory props range ------"
csvtk pretty csvs/ap-nl-dataset-mand-props.csv


['---- DCAT-AP-NL - mandatory props range ------',
 'prop_path                                prop_range                                    prop_shape                                                                                                 ',
 '--------------------------------------   -------------------------------------------   -----------------------------------------------------------------------------------------------------------',
 'http://purl.org/dc/terms/description                                                   https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DatasetShape/8f61614144aa7bca188b24f5976593dc08aad0e6',
 'http://purl.org/dc/terms/title                                                         https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DatasetShape/f8b02efd063b8089b72ce9677a1e4a3488eeb9a9',
 'http://purl.org/dc/terms/accessRights    http://purl.org/dc/terms/RightsStatement      http://modellen.geostandaarden.nl/dcat-ap-nl/id/shape/DatasetSha

# Controlled Vocabularies Constraints

**[csvs/ap-nl-dataset-CVs.csv](csvs/ap-nl-dataset-CVs.csv)**  Overview of DCAT-AP Dataset properties with controlled vocs as range

[dcat-ap/releases/3.0.1/html/shacl/mdr-vocabularies.shape.ttl](dcat-ap/releases/3.0.1/html/shacl/mdr-vocabularies.shape.ttl) specifies the controlled vocabulary constraints on properties expressed by DCAT-AP in SHACL.

More info in [DCAT-AP Documentation on CVs](https://semiceu.github.io/DCAT-AP/releases/3.0.1/#controlled-vocabularies-to-be-used)

## Data theme CV
From the mandatory Dataset properties, only dcat:theme has a skos:Concept as range. 
The advised vocabulary to use is the "Data theme" http://publications.europa.eu/resource/authority/data-theme (view [human-readable interface](https://op.europa.eu/web/eu-vocabularies/concept-scheme/-/resource?uri=http://publications.europa.eu/resource/authority/data-theme)). Where the candidate concept for SSH/ODISSEI seems to be :
* [SOCI](http://publications.europa.eu/resource/authority/data-theme/SOCI) Population and society

Other possible themes are [ECON](https://op.europa.eu/en/web/eu-vocabularies/concept/-/resource?uri=http://publications.europa.eu/resource/authority/data-theme/ECON) Economy and finance & [EDUC](https://op.europa.eu/web/eu-vocabularies/concept/-/resource?uri=http://publications.europa.eu/resource/authority/data-theme/EDUC) Education, culture and sport, but these seem to be too specific and not matching the SSH DS and ODISSEI Portal domains. https://op.europa.eu/en/web/eu-vocabularies/concept-scheme/-/resource?uri=http://publications.europa.eu/resource/authority/data-theme contains more detailed info 




* start query at :Dataset_ShapeCV
* query sh:property  sh:path/sh:nodeKind/sh:description



In [322]:

sparql_dataset_props_cvs = '''
SELECT ?dataset_prop ?voc_desc
WHERE {
       <http://data.europa.eu/r5r#Dataset_ShapeCV> sh:property ?shape_prop .
       ?shape_prop sh:path ?dataset_prop ;
                   sh:description ?voc_desc  
          
}''' 


ap_dataset_props_CVs = sparql_files_query(query=sparql_dataset_props_cvs, 
                                                          format='ttl', 
                                                          filespath=['dcat-ap/releases/3.0.1/html/shacl/mdr-vocabularies.shape.ttl'])
for result in ap_dataset_props_CVs:
    print(result)

ap_dataset_props_CVs_list = [item.asdict() for item in ap_dataset_props_CVs]
print(ap_dataset_props_CVs_list)
create_csv(filepath='csvs/ap-nl-dataset-CVs.csv',
           headers=['dataset_prop', 'voc_desc'],
           data_dict=ap_dataset_props_CVs_list
           )    

(rdflib.term.URIRef('http://purl.org/dc/terms/accrualPeriodicity'), rdflib.term.Literal('A non EU managed concept is used to indicate the accrualPeriodicity frequency. If no corresponding can be found inform the maintainer of the EU frequency NAL'))
(rdflib.term.URIRef('http://purl.org/dc/terms/language'), rdflib.term.Literal('A non EU managed concept is used to indicate a language. If no corresponding can be found inform the maintainer of the EU language NAL'))
(rdflib.term.URIRef('http://purl.org/dc/terms/publisher'), rdflib.term.Literal('A non EU managed concept is used to indicate the publisher, check if a corresponding exists in the EU corporates bodies NAL'))
(rdflib.term.URIRef('http://purl.org/dc/terms/spatial'), rdflib.term.Literal('A non managed concept is used to indicate a spatial description, check if a corresponding exists'))
(rdflib.term.URIRef('http://www.w3.org/ns/dcat#theme'), rdflib.term.Literal('Multiple themes can be used but at least one concept of <http://publica

# Recommended properties of Dataset

in [csvs/ap-nl-dataset-recommended-props.csv](csvs/ap-nl-dataset-recommended-props.csv)´

**TODO:**
* study them closely
* include relevant ones in requirements doc

In [323]:
# Goal: recommended Dataset properties in DCAT-AP & DCAT-AP-NL 


sparql_dataset_rec_props = '''
SELECT ?dataset_prop ?profile
WHERE {
        { 
            BIND('ap' AS ?profile)
            <http://data.europa.eu/r5r#Dataset_Shape> sh:property ?shape_prop .
            ?shape_prop sh:path ?dataset_prop .
        }
        UNION
        { 
            BIND('ap-NL' AS ?profile)
            <http://modellen.geostandaarden.nl/dcat-ap-nl/id/shape/DatasetShape_aanbevolen> sh:property ?shape_prop .
            ?shape_prop sh:path ?dataset_prop .
        }        
}''' 


ap_dataset_recommended_props = sparql_files_query(query=sparql_dataset_rec_props, 
                                                          format='ttl', 
                                                          filespath=['dcat-ap/releases/3.0.1/html/shacl/shapes_recommended.ttl',
                                                                     'dcat-ap-nl/shapes/dcat-ap-nl-SHACL-aanbevolen.ttl'
                                                                     ])
for result in ap_dataset_recommended_props:
    print(result)

ap_dataset_recommended_props_list = [item.asdict() for item in ap_dataset_recommended_props]
create_csv(filepath='csvs/ap-nl-dataset-recommended-props.csv',
           headers=['dataset_prop', 'profile'],
           data_dict=ap_dataset_recommended_props_list
           )   

(rdflib.term.URIRef('http://www.w3.org/ns/dcat#contactPoint'), rdflib.term.Literal('ap'))
(rdflib.term.URIRef('http://www.w3.org/ns/dcat#distribution'), rdflib.term.Literal('ap'))
(rdflib.term.URIRef('http://www.w3.org/ns/dcat#keyword'), rdflib.term.Literal('ap'))
(rdflib.term.URIRef('http://purl.org/dc/terms/publisher'), rdflib.term.Literal('ap'))
(rdflib.term.URIRef('http://purl.org/dc/terms/spatial'), rdflib.term.Literal('ap'))
(rdflib.term.URIRef('http://purl.org/dc/terms/temporal'), rdflib.term.Literal('ap'))
(rdflib.term.URIRef('http://www.w3.org/ns/dcat#theme'), rdflib.term.Literal('ap'))
(rdflib.term.URIRef('http://purl.org/dc/terms/conformsTo'), rdflib.term.Literal('ap-NL'))
(rdflib.term.URIRef('http://xmlns.com/foaf/0.1/page'), rdflib.term.Literal('ap-NL'))
(rdflib.term.URIRef('http://www.w3.org/ns/dcat#landingPage'), rdflib.term.Literal('ap-NL'))
(rdflib.term.URIRef('http://purl.org/dc/terms/language'), rdflib.term.Literal('ap-NL'))


# Supportive Entities

## dcat:Distribution
Dataset recommended prop dcat:distribution 

> A physical embodiment of the Dataset in a particular format. 


In [374]:

sparql_distrib_shapes = ''' 
DESCRIBE ?shape ?shape_prop
WHERE{
    BIND( dcat:Distribution AS ?targetClass)
    ?shape sh:targetClass ?targetClass ;
            sh:property ?shape_prop .     
}
'''
result_distrib_shapes = sparql_files_query(query=sparql_distrib_shapes,
                                           format='ttl', 
                                           filespath=[
                                               'dcat-ap/releases/3.0.1/shacl/dcat-ap-SHACL.ttl',
                                               'dcat-ap-nl/shapes/dcat-ap-nl-SHACL.ttl'
                                                      ])
print(result_distrib_shapes.serialize(format='ttl').decode('utf-8'))

@prefix adms: <http://www.w3.org/ns/adms#> .
@prefix dcat: <http://www.w3.org/ns/dcat#> .
@prefix dcatap: <http://data.europa.eu/r5r/> .
@prefix dcatapnl-sh: <http://modellen.geostandaarden.nl/dcat-ap-nl/id/shape/> .
@prefix dct: <http://purl.org/dc/terms/> .
@prefix eli: <http://data.europa.eu/eli/ontology#> .
@prefix eush: <https://purl.eu/ns/shacl#> .
@prefix foaf: <http://xmlns.com/foaf/0.1/> .
@prefix odrl: <http://www.w3.org/ns/odrl/2/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix skos: <http://www.w3.org/2004/02/skos/core#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

dcatapnl-sh:DistributionShape a sh:NodeShape ;
    sh:name "Distribution"@en ;
    sh:property dcatapnl-sh:DistributionShape_accessService_maxCount,
        dcatapnl-sh:DistributionShape_accessURL_maxCount,
        dcatapnl-sh:DistributionShape_applicableLegislation_nodeKind,
        dcatapnl-sh:DistributionShape_downloadURL_maxCount,
       

# Distribution

OProfiles documentation:
* https://semiceu.github.io/DCAT-AP/releases/3.0.1/#Distribution
* https://docs.geostandaarden.nl/dcat/dcat-ap-nl30/#distribution-dcat-distribution

![img/dcatap-NL-DistributionSHACL.svg](img/dcatap-NL-DistributionSHACL.svg) 
image: dcat:Distribution SHACL shapes, based on above query ,rendered by [https://shacl-play.sparna.fr/play/](https://shacl-play.sparna.fr/play/). 




**Mandatory properties:**
* dcat:accessURL (DCAT-AP)
* dct:license (DCAT-AP-NL)

**Interesting (optional) properties for DANS:** 
* dct:issue (data property) The date of formal issuance (e.g., publication)
* http://spdx.org/rdf/terms#checksum (object property) [More on Checksum class](https://semiceu.github.io/DCAT-AP/releases/3.0.1/#Checksum)
    * algorithm = SHA1  (used by Dataverse)
    * checksum value 
* dct:format (object property) - Although [dct:MediaTypeOrExtent](https://www.dublincore.org/specifications/dublin-core/dcmi-terms/#MediaTypeOrExtent) & [dct:MediaType](https://www.dublincore.org/specifications/dublin-core/dcmi-terms/#MediaType) classes to not offer info class properties


In [ ]:
# Goal: dcat:Distributions properties ranges



sparql_distrib_props = '''
SELECT DISTINCT  ?prop ?min ?max
WHERE {
        {
            BIND( dcat:Distribution AS ?targetClass)
            ?dist_shape sh:targetClass ?targetClass ; 
                        sh:property ?shape_prop .
            ?shape_prop sh:path ?prop ;
                        sh:name ?prop_name .
            OPTIONAL { ?shape_prop sh:minCount ?min }
            OPTIONAL { ?shape_prop sh:maxCount ?max }

        }

        # UNION
        # {
        #     <https://semiceu.github.io/DCAT-AP/releases/3.0.1/shacl/dcat-ap-SHACL.ttl#dcat:DistributionShape> sh:property ?shape_prop_ .
        #     ?shape_prop_ sh:path ?prop .
        #     FILTER NOT EXISTS { ?shape_prop_ sh:class ?prop_range_ } . 
        # }


            # OPTIONAL {
            #     dcatapnl-sh:DistributionShape sh:property ?nl_shape_prop .
            #     ?nl_shape_prop sh:path ?prop
            #     }
}
ORDER BY ?prop_name
''' 

ap_distrib = sparql_files_query(query=sparql_distrib_props, 
                                                          format='ttl', 
                                                          filespath=['dcat-ap/releases/3.0.1/shacl/dcat-ap-SHACL.ttl',
                                                                     'dcat-ap-nl/shapes/dcat-ap-nl-SHACL.ttl'])
for result in ap_distrib:
    print(result)


# create_csv(filepath='csvs/ap-nl-distribution-overview.csv',
#            headers=['targetClass', 'prop', 'prop_range'],
#            data_dict=[item.asdict() for item in ap_distrib]
#            )   

(rdflib.term.URIRef('http://www.w3.org/ns/dcat#accessURL'), None, None)
(rdflib.term.URIRef('http://www.w3.org/ns/dcat#accessURL'), rdflib.term.Literal('1', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')), None)
(rdflib.term.URIRef('http://www.w3.org/ns/dcat#accessURL'), None, rdflib.term.Literal('1', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))
(rdflib.term.URIRef('http://www.w3.org/ns/dcat#accessService'), None, None)
(rdflib.term.URIRef('http://www.w3.org/ns/dcat#accessService'), None, rdflib.term.Literal('1', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))
(rdflib.term.URIRef('http://data.europa.eu/r5r/applicableLegislation'), None, None)
(rdflib.term.URIRef('http://data.europa.eu/r5r/availability'), None, None)
(rdflib.term.URIRef('http://data.europa.eu/r5r/availability'), None, rdflib.term.Literal('1', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))
(rdflib.term.URIRef('http:

In [343]:
# dcat-ap-nl Distribution shapes

describe_distr_sh_NL = '''
DESCRIBE ?shape_prop
WHERE {

           dcatapnl-sh:DistributionShape sh:property ?shape_prop .

}
ORDER BY ?shape_prop
'''


describe_distr_sh_NL_results = sparql_files_query(query=describe_distr_sh_NL, 
                                format='ttl', 
                                filespath=['dcat-ap-nl/shapes/dcat-ap-nl-SHACL.ttl']
                                )
                                           
print(describe_distr_sh_NL_results.serialize(format='ttl').decode('utf-8'))




@prefix dcat: <http://www.w3.org/ns/dcat#> .
@prefix dcatap: <http://data.europa.eu/r5r/> .
@prefix dcatapnl-sh: <http://modellen.geostandaarden.nl/dcat-ap-nl/id/shape/> .
@prefix dct: <http://purl.org/dc/terms/> .
@prefix eush: <https://purl.eu/ns/shacl#> .
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

dcatapnl-sh:DistributionShape_accessService_maxCount sh:maxCount 1 ;
    sh:name "access service"@en ;
    sh:path dcat:accessService ;
    eush:message "Maximally 1 values are allowed for access service"@en .

dcatapnl-sh:DistributionShape_accessURL_maxCount sh:maxCount 1 ;
    sh:name "access URL"@en ;
    sh:path dcat:accessURL ;
    eush:message "Maximally 1 values are allowed for access URL"@en .

dcatapnl-sh:DistributionShape_applicableLegislation_nodeKind sh:name "applicable legislation"@en ;
    sh:nodeKind sh:BlankNodeOrIRI ;
    sh:path dcatap:applicableLegislation ;
    eush:message "The expected value for applicable legislati

In [341]:
# Goal: dcat:Distributions properties ranges
# OUTPUT
# Goal: list of all dcat:Dataset mandatory properties in DCAT-AP & DCAT-APN-NL
# shacl:minCount 1

# https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DistributionShape/0a6f3bb11ed4ea12f852c78996b89c9a54ffc0fb
# https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DistributionShape/73465b7fbd7f991a08ddd1b766c2e46fa9dfc14e 

describe_distr_prop = ''' 
DESCRIBE <https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DistributionShape/0a6f3bb11ed4ea12f852c78996b89c9a54ffc0fb>
<https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DistributionShape/73465b7fbd7f991a08ddd1b766c2e46fa9dfc14e>
'''

describe_distr_prop = '''
DESCRIBE ?shape_prop
WHERE {

            <https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DistributionShape> sh:property ?shape_prop .
            ?shape_prop sh:path ?sh_path ;
                        sh:name ?sh_name .
}
ORDER BY ?sh_name
'''



ap_distrib_ttl = sparql_files_query(query=describe_distr_prop, 
                                format='ttl', 
                                filespath=['dcat-ap/releases/3.0.1/shacl/dcat-ap-SHACL.ttl',]
                                )
                                            #'dcat-ap-nl/shapes/dcat-ap-nl-SHACL.ttl']
                                             


print(ap_distrib_ttl.serialize(format='ttl').decode('utf-8'))






# create_csv(filepath='dcat-ap-nl_mand_props.csv',
#            headers=ap_NL_mandatory_dataset_props_list[0].keys(),
#            data_dict=ap_NL_mandatory_dataset_props_list)

# TODO: dcat-ap/releases/3.0.0/shacl/ranges.ttl


# create_csv(filepath='csvs/ap-nl-distribution-overview.csv',
#            headers=['targetClass', 'prop', 'prop_range'],
#            data_dict=[item.asdict() for item in ap_distrib]
#            )   


@prefix dc1: <http://purl.org/dc/terms/> .
@prefix dcat: <http://www.w3.org/ns/dcat#> .
@prefix foaf: <http://xmlns.com/foaf/0.1/> .
@prefix odrl: <http://www.w3.org/ns/odrl/2/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix shacl: <http://www.w3.org/ns/shacl#> .
@prefix skos: <http://www.w3.org/2004/02/skos/core#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DistributionShape/06ed12d1071a78728d10628eb2d072781559eb75> rdfs:seeAlso "https://semiceu.github.io/DCAT-AP/releases/3.0.1#Distribution.mediatype" ;
    shacl:class dc1:MediaType ;
    shacl:description "The media type of the Distribution as defined in the official register of media types managed by IANA."@en ;
    shacl:name "media type"@en ;
    shacl:path dcat:mediaType .

<https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DistributionShape/0796df7f5520ac6500217e81e434f9a755f64afe> rdfs:seeAlso "https://semiceu.github.io/DCAT-AP/releases/3.0.


<https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DistributionShape/0a6f3bb11ed4ea12f852c78996b89c9a54ffc0fb> rdfs:seeAlso "https://semiceu.github.io/DCAT-AP/releases/3.0.1#Distribution.applicablelegislation";
  shacl:description "The legislation that mandates the creation or management of the Distribution."@en;
  shacl:name "applicable legislation"@en;
  shacl:nodeKind shacl:BlankNodeOrIRI;
  shacl:path <http://data.europa.eu/r5r/applicableLegislation> .



<https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DistributionShape/73465b7fbd7f991a08ddd1b766c2e46fa9dfc14e> rdfs:seeAlso "https://semiceu.github.io/DCAT-AP/releases/3.0.1#Distribution.applicablelegislation";
    shacl:class <http://data.europa.eu/eli/ontology#LegalResource>;
  shacl:description "The legislation that mandates the creation or management of the Distribution."@en;
  shacl:name "applicable legislation"@en;
  shacl:path <http://data.europa.eu/r5r/applicableLegislation> .


In [326]:
# Output:  for DCAT-AP-NL Distribution mandatory properties and their range (sh:class ?prop_range) 
# Output: ap-nl-dataset-mand-props.csv


sparql_dcat_mandatory_distribution_props_ranges_vars = '''
SELECT  ?prop_path  ?prop_range
WHERE {
    BIND( dcat:Distribution AS ?targetClass )
    VALUES ?prop_path { %s } 
    ?shape a sh:NodeShape ;
           sh:targetClass ?targetClass ;
           sh:property ?prop_shape_uri .
    ?prop_shape_uri sh:path ?prop_path ;
                    sh:class ?prop_range .
    
}''' % prop_path_for_sparql_str4query


ap_NL_mandatory_dataset_props_range = sparql_files_query(query=sparql_dcat_mandatory_distribution_props_ranges_vars, 
                                                          format='ttl', 
                                                          filespath=['dcat-ap/releases/3.0.1/shacl/ranges.ttl'])
ap_NL_mandatory_dataset_props_range_list = [row.asdict() for row in ap_NL_mandatory_dataset_props_range]

# Join  ap_NL_mandatory_dataset_props_range_list & ap_NL_mandatory_dataset_props_list
# Step 1: Build a mapping from prop_path to prop_range
prop_range_map = {d['prop_path']: d['prop_range'] for d in ap_NL_mandatory_dataset_props_range_list}
# Step 2: Merge the lists
ap_NL_mandatory_dataset_props_merged = []
for d in ap_NL_mandatory_dataset_props_list:
    # Copy to avoid mutating the original
    merged_dict = d.copy()
    prop_path = d['prop_path']
    if prop_path in prop_range_map:
        merged_dict['prop_range'] = prop_range_map[prop_path]
    ap_NL_mandatory_dataset_props_merged.append(merged_dict)
print(f"{'*'*3} Output of ap_NL_mandatory_dataset_props_merged in ap-nl-dataset-mand-props.csv {'*'*3}\n")
# print(ap_NL_mandatory_dataset_props_merged)
create_csv(filepath='csvs/ap-nl-dataset-mand-props.csv',
           headers=['prop_path', 'prop_range', 'prop_shape'],
           data_dict=ap_NL_mandatory_dataset_props_merged)


*** Output of ap_NL_mandatory_dataset_props_merged in ap-nl-dataset-mand-props.csv ***



## other focal points:


classes supporting Dataset:
* from prop dct:accessRights  dct:RightsStatement
* from prop dcat:contactPoint vcard:Kind
* from prop dct:creator, dct:publisher foaf:Agent
* from prop dcat:distribution dcat:Distribution (not mandatory, but essential for DANS)

look into foaf:Agent, dcat:Distribution